In [ ]:
import os
from os import path
import random
import pickle

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from scipy import stats

In [ ]:
data_folder = './data'
output_folder = './output'
if not path.exists(output_folder):
    os.makedirs(output_folder)

In [ ]:
with open(path.join(data_folder, 'users.pickle'), 'rb') as f:
    users = pickle.load(f)

with open(path.join(data_folder, 'daily_counts.pickle'), 'rb') as f:
    daily_counts = pickle.load(f)    

In [ ]:
# Utility functions

def plot_and_calc_corr(df, v1, v2, title):
    df_clean = df.dropna(subset=[v1, v2])
    plt.hist2d(df_clean[v1], df_clean[v2], cmap='Reds')
    plt.title(title)

    spearman_corr, spearman_p = stats.spearmanr(df_clean[v1], df_clean[v2])

    print(f"Spearman Correlation: {spearman_corr:.4f}")
    print(f"P-value: {spearman_p:.4f}")

    plt.savefig(path.join(output_folder, title + '.pdf'), bbox_inches='tight')

def compare_dists(data1, data2, label1, label2):
    data1.hist(label=label1, alpha=0.5)
    data2.hist(label=label2, alpha=0.5)
    
    plt.legend() 
    plt.title("Distribution Comparison")
    plt.show()

    ks_stat, ks_p = stats.ks_2samp(data1, data2)
    print(f"KS Test: statistic={ks_stat:.4f}, p-value={ks_p:.4f}")

    mw_stat, mw_p = stats.mannwhitneyu(data1, data2)
    print(f"Mann-Whitney U: statistic={mw_stat:.4f}, p-value={mw_p:.4f}")

    res = stats.anderson_ksamp([data1, data2], method=stats.PermutationMethod())
    print(f"Anderson-Darling: statistic={res.statistic:.4f}, p-value={res.pvalue:.4f}")

    ws_dist = stats.wasserstein_distance(data1, data2)
    print(f"Wasserstein Distance: {ws_dist:.4f}")

## Descriptive plots

In [ ]:
maxc = 20
plt.figure(figsize=(8, 5))
plt.title('Daily cases')

plt.plot(daily_counts['date'], daily_counts['infections'], color='red')

plt.xticks(rotation=45, ha='right')
plt.ylim(0, maxc)
plt.ylabel('Daily count')
    
plt.savefig(path.join(output_folder, 'daily-cases.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
behaviors = ['quarantine_yes', 'quarantine_no']
labels = ['Quarantined', 'Not quarantined']
maxc = 250
plt.figure(figsize=(8, 5))
plt.title('Daily quarantine')

for b, t in zip(behaviors, labels):
    plt.plot(daily_counts['date'], daily_counts[b], label=t)

plt.xticks(rotation=45, ha='right')
plt.ylim(0, maxc)
plt.ylabel('Daily count')

plt.legend()
    
plt.savefig(path.join(output_folder, 'daily-quarantine.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
topic_map = {
 'S1_Q1': 'Real-life susceptibility', 
 'S1_Q2': 'Real-life severity', 
 'S1_Q3': 'Real-life self-efficacy', 
 'S1_Q4': 'Real-life benefit', 
 'S1_Q5': 'Gender', 
 'S1_Q6': 'School affiliation', 
 'S2_Q1': 'In-game susceptibility', 
 'S2_Q2': 'In-game severity', 
 'S2_Q3': 'In-game self-efficacy', 
 'S2_Q4': 'In-game benefit',
 'S3_Q1': 'Post-game realism', 
 'S3_Q2': 'Post-game concern', 
 'S3_Q3': 'Post-game reflection', 
 'S3_Q4': 'Post-game understanding',
 'S3_Q5': 'Field of study'
}

gender_labels = ['Male', 'Female', 'Other', 'NA']
gender_ticks = [1, 2, 3, 4]

affiliation_labels = ['Student', 'Faculty', 'Staff', 'Other', 'NA']
affiliation_ticks = [1, 2, 3, 4, 5]

field_labels = ['Bio', 'Law', 'Edu', 'Biz', 'Eng', 'CS']
field_ticks = [1, 2, 3, 4, 5, 6]

for var in list(topic_map.keys()):
    # 1. Get min and max to define the range
    min_val = int(users[var].min())
    max_val = int(users[var].max())
    
    # 2. Define bins at half-integers (e.g., 0.5, 1.5, ... 6.5)
    # We add +2 to max_val to ensure the right edge of the last bin is included
    bins = np.arange(min_val, max_val + 2) - 0.5
    
    # 3. Plot
    # rwidth=0.8 creates the gap (bars use only 80% of the bin width)
    if 'S1' in var:
        col = 'tab:blue'
    elif 'S2' in var:
        col = 'tab:purple'
    else:    
        col = 'tab:green'
    users.hist(column=var, bins=bins, rwidth=0.8, grid=False, color=col)
    plt.title(var + ' - ' + topic_map[var])

    if var == 'S1_Q5':
        plt.xticks(gender_ticks, gender_labels)
    elif var == 'S1_Q6':
        plt.xticks(affiliation_ticks, affiliation_labels)
    elif var == 'S3_Q5':
        plt.xticks(field_ticks, field_labels)        
    else:
        plt.xticks(range(min_val, max_val + 1))
    
    plt.savefig(path.join(output_folder, var + '.pdf'), bbox_inches='tight')
    plt.show()

## Statistical analyses of groups and survey responses

In [ ]:
users_g1 = users[users['group'] == 1]
len(users_g1)

In [ ]:
users_g2 = users[users['group'] == 2]
len(users_g2)

In [ ]:
compare_dists(users_g1['no_quarantine'], users_g2['no_quarantine'], 'G1', 'G2')

In [ ]:
plot_and_calc_corr(users, 'S1_Q1', 'no_quarantine', 'S1_Q1-no_quarantine')

In [ ]:
plot_and_calc_corr(users, 'S1_Q2', 'no_quarantine', 'S1_Q2-no_quarantine')

In [ ]:
plot_and_calc_corr(users, 'S1_Q3', 'no_quarantine', 'S1_Q3+no_quarantine')

In [ ]:
plot_and_calc_corr(users, 'S1_Q4', 'no_quarantine', 'S1_Q4+no_quarantine')

In [ ]:
plot_and_calc_corr(users, 'S1_Q5', 'no_quarantine', 'S1_Q5+no_quarantine')

In [ ]:
plot_and_calc_corr(users, 'S1_Q5', 'quarantine', 'S1_Q5+quarantine')

In [ ]:
plot_and_calc_corr(users_g1, 'S1_Q5', 'quarantine', 'S1_Q5+quarantine-G1')

In [ ]:
plot_and_calc_corr(users_g2, 'S1_Q5', 'quarantine', 'S1_Q5+quarantine-G2')

## Correlation between real-life and game attitudes

In [ ]:
plot_and_calc_corr(users, 'S1_Q1', 'S2_Q1', 'Susceptibility (Q1)')

In [ ]:
plot_and_calc_corr(users, 'S1_Q2', 'S2_Q2', 'Severity (Q2)')

In [ ]:
plot_and_calc_corr(users, 'S1_Q3', 'S2_Q3', 'Self-efficacy (Q3)')

In [ ]:
plot_and_calc_corr(users, 'S1_Q4', 'S2_Q4', 'Benefit (Q4)')